# Newton's Second Law

This version keeps each experiment self-contained: the controls and animation appear together.

It uses ordinary Matplotlib objects only (`Rectangle`, `FancyArrowPatch`, `ax.text()`) and renders the motion as an in-browser animation.

## 1. Motion demo

Change $F$ and $m$. The acceleration is

$$a=\frac{F}{m}$$

Press **Run** to watch the block move.

In [ ]:
import numpy as np

import matplotlib.pyplot as plt

from matplotlib.patches import Rectangle, FancyArrowPatch

from matplotlib.animation import FuncAnimation

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    %pip install -q ipywidgets
    import ipywidgets as widgets

from IPython.display import display, HTML





def build_block_figure(x=0.0, F=100, m=20, a=5.0):

    """Draw the block, the force arrow and the acceleration arrow at position x."""

    fig, ax = plt.subplots(figsize=(8, 3.2))

    ax.set_xlim(0, 25)

    ax.set_ylim(-1, 6)

    ax.axhline(1, color="black", linewidth=2)



    block = Rectangle((x, 1), 2.5, 1.6, facecolor="lightblue",

                      edgecolor="black", linewidth=2)

    ax.add_patch(block)

    mass_text = ax.text(x + 1.25, 1.8, f"{m} kg", ha="center", va="center", fontsize=13)



    force_arrow = FancyArrowPatch((x + 2.7, 1.8), (x + 5.7, 1.8),

                                  arrowstyle="-|>", mutation_scale=20,

                                  linewidth=2, color="black")

    ax.add_patch(force_arrow)

    force_text = ax.text(x + 4.2, 2.25, f"F = {F} N", ha="center", fontsize=12)



    accel_arrow = FancyArrowPatch((x + 0.2, 4.1), (x + 4.2, 4.1),

                                  arrowstyle="-|>", mutation_scale=22,

                                  linewidth=2.5, color="firebrick")

    ax.add_patch(accel_arrow)

    accel_text = ax.text(x + 2.2, 4.65, f"a = {a:.2f} m/s²",

                         ha="center", fontsize=13, fontweight="bold")



    ax.set_title("Block motion")

    ax.set_xlabel("Position")

    ax.set_yticks([])

    for side in ("left", "right", "top"):

        ax.spines[side].set_visible(False)

    fig.tight_layout()



    artists = dict(

        block=block, mass_text=mass_text,

        force_arrow=force_arrow, force_text=force_text,

        accel_arrow=accel_arrow, accel_text=accel_text,

    )

    return fig, artists





def block_motion_animation(F, m):

    """Build an in-browser animation of the block accelerating under force F."""

    a = F / m

    times = np.linspace(0, 2.0, 35)

    xs = [min(21.5, 0.5 * a * t**2) for t in times]



    fig, art = build_block_figure(x=0.0, F=F, m=m, a=a)



    def update(i):

        x = xs[i]

        art["block"].set_x(x)

        art["mass_text"].set_position((x + 1.25, 1.8))

        art["force_arrow"].set_positions((x + 2.7, 1.8), (x + 5.7, 1.8))

        art["force_text"].set_position((x + 4.2, 2.25))

        art["accel_arrow"].set_positions((x + 0.2, 4.1), (x + 4.2, 4.1))

        art["accel_text"].set_position((x + 2.2, 4.65))

        return tuple(art.values())



    anim = FuncAnimation(fig, update, frames=len(xs), interval=60, repeat=False)

    plt.close(fig)

    return HTML(anim.to_jshtml())

In [ ]:
# ============================================================

# DEMO 1: vary F and m, then watch the block move

# ============================================================



force_slider = widgets.IntSlider(

    value=100, min=20, max=300, step=20,

    description="Force F (N):",

    continuous_update=False

)



mass_slider = widgets.IntSlider(

    value=20, min=5, max=60, step=5,

    description="Mass m (kg):",

    continuous_update=False

)



run_button = widgets.Button(description="Run", button_style="primary", icon="play")

reset_button = widgets.Button(description="Reset")

value_label = widgets.HTML()

sim_output = widgets.Output(layout=widgets.Layout(width="700px"))





def show_motion_value():

    F = force_slider.value

    m = mass_slider.value

    a = F / m

    value_label.value = (

        f"<b>F = {F} N</b><br>"

        f"<b>m = {m} kg</b><br>"

        f"<b>a = {a:.2f} m/s²</b>"

    )





def show_motion_initial():

    F = force_slider.value

    m = mass_slider.value

    a = F / m

    show_motion_value()

    fig, _ = build_block_figure(x=0.0, F=F, m=m, a=a)

    display(fig)

    plt.close(fig)





def run_clicked(button):

    show_motion_value()

    with sim_output:

        sim_output.clear_output(wait=True)

        display(block_motion_animation(force_slider.value, mass_slider.value))





def reset_clicked(button):

    with sim_output:

        sim_output.clear_output(wait=True)

        show_motion_initial()





run_button.on_click(run_clicked)

reset_button.on_click(reset_clicked)



controls = widgets.VBox([

    widgets.HTML("<h3>Controls</h3>"),

    force_slider,

    mass_slider,

    widgets.HBox([run_button, reset_button]),

    value_label

], layout=widgets.Layout(width="320px"))



display(widgets.HBox([controls, sim_output],

                     layout=widgets.Layout(align_items="flex-start", gap="18px")))



with sim_output:

    show_motion_initial()

## 2. Build an $F$ versus $a$ graph

For this experiment the mass is fixed.

Change only $F$, press **Run + record**, and collect the pair $(a,F)$.

Because

$$F=ma$$

a plot of $F$ against $a$ should be a straight line. Its slope is the mass.

In [ ]:
# ============================================================

# DEMO 2: fixed mass, vary F, record (a, F) pairs

# ============================================================



MASS = 20  # kg -- students can change this one number



force_exp = widgets.IntSlider(

    value=200, min=20, max=300, step=20,

    description="Force F (N):",

    continuous_update=False

)



run_record = widgets.Button(description="Run + record", button_style="primary", icon="play")

reset_exp = widgets.Button(description="Reset block")

clear_data = widgets.Button(description="Clear data")

exp_value_label = widgets.HTML()



exp_output = widgets.Output(layout=widgets.Layout(width="700px"))

graph_output = widgets.Output(layout=widgets.Layout(width="760px"))

show_fit = widgets.Checkbox(value=False, description="Show fit line")



F_values = []

a_values = []





def show_exp_value():

    F = force_exp.value

    a = F / MASS

    exp_value_label.value = (

        f"<b>Fixed mass = {MASS} kg</b><br>"

        f"<b>F = {F} N</b><br>"

        f"<b>a = {a:.2f} m/s²</b>"

    )





def show_exp_initial():

    F = force_exp.value

    a = F / MASS

    show_exp_value()

    fig, _ = build_block_figure(x=0.0, F=F, m=MASS, a=a)

    display(fig)

    plt.close(fig)





def draw_F_a_graph():

    with graph_output:

        graph_output.clear_output(wait=True)



        fig2, ax2 = plt.subplots(figsize=(7, 4.5))

        ax2.scatter(a_values, F_values, s=70)



        if show_fit.value and len(a_values) >= 2:

            slope, intercept = np.polyfit(a_values, F_values, 1)

            xmax = max(a_values) * 1.12

            xx = np.linspace(0, xmax, 100)

            ax2.plot(xx, slope * xx + intercept)

            ax2.text(

                0.05, 0.92,

                f"slope = {slope:.1f} kg",

                transform=ax2.transAxes,

                fontsize=12

            )



        ax2.set_xlim(left=0)

        ax2.set_ylim(bottom=0)

        ax2.set_xlabel("Acceleration, a (m/s²)")

        ax2.set_ylabel("Force, F (N)")

        ax2.set_title("F versus a")

        ax2.grid(alpha=0.3)

        plt.show()

        plt.close(fig2)





def run_record_clicked(button):

    show_exp_value()

    with exp_output:

        exp_output.clear_output(wait=True)

        display(block_motion_animation(force_exp.value, MASS))



    # Record exactly one pair at the end of the run.

    a_values.append(force_exp.value / MASS)

    F_values.append(force_exp.value)

    draw_F_a_graph()





def reset_exp_clicked(button):

    with exp_output:

        exp_output.clear_output(wait=True)

        show_exp_initial()





def clear_data_clicked(button):

    F_values.clear()

    a_values.clear()

    draw_F_a_graph()





run_record.on_click(run_record_clicked)

reset_exp.on_click(reset_exp_clicked)

clear_data.on_click(clear_data_clicked)

show_fit.observe(lambda change: draw_F_a_graph(), names="value")



exp_controls = widgets.VBox([

    widgets.HTML("<h3>Controls</h3>"),

    force_exp,

    widgets.HBox([run_record, reset_exp]),

    clear_data,

    show_fit,

    exp_value_label

], layout=widgets.Layout(width="320px"))



with exp_output:

    show_exp_initial()

draw_F_a_graph()



display(widgets.VBox([

    widgets.HBox([exp_controls, exp_output],

                 layout=widgets.Layout(align_items="flex-start", gap="18px")),

    graph_output

]))